In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np

In [ ]:
# Load layers
buildings = gpd.read_file("../layers/bcn_catastral_building.gpkg")
noise_streets = gpd.read_file("../layers/BCN_noise_streets.gpkg")

# Ensure CRS is the same
buildings = buildings.to_crs(noise_streets.crs)
display(buildings.head(3))

In [ ]:
# Process the number of floors
def clean_floors_col(series):
    # Remove non-numeric characters and convert to float
    return pd.to_numeric(series.astype(str).str.replace(r'[^\d.]', '', regex=True), errors='coerce')

# Initialize a series for estimated floors
floors = clean_floors_col(buildings['numberOfFloorsAboveGround'])

# Impute missing values with the median number of floors
median_floors = floors.median()
if np.isnan(median_floors):
    median_floors = 3.0  # Fallback to 3 stories if dataset is too sparse
    
floors = floors.fillna(median_floors)
buildings['estimated_floors'] = floors

print(f"All buildings have assigned floors. Median is: {median_floors:.1f}")

fig, ax = plt.subplots(figsize=(6, 4))
buildings['estimated_floors'].plot(kind='hist', bins=20, ax=ax, color='lightgreen', edgecolor='black')
ax.set_title('Distribution of Number of Floors Above Ground')
ax.set_xlabel('Number of Floors')
ax.set_xlim(0, 20)  # Focus on the majority
plt.show()

In [ ]:
def calculate_building_floors_in_buffer(streets_gdf, buildings_gdf, buffer_size):
    print(f"Evaluating {buffer_size}m buffers...")
    # Buffer the streets
    buffered_streets = streets_gdf.copy()
    buffered_streets['geometry'] = buffered_streets.geometry.buffer(buffer_size)
    
    # Keep only target columns for spatial join to reduce memory usage
    bldgs = buildings_gdf[['estimated_floors', 'geometry']].copy()
    
    # Spatial join: Which buildings intersect which street buffer?
    joined = gpd.sjoin(buffered_streets[['TRAM', 'geometry']], bldgs, how='left', predicate='intersects')
    
    # If a street buffer hits nothing, it will have NaN in estimated_floors. Treatment: 0.
    joined['estimated_floors'] = joined['estimated_floors'].fillna(0)
    
    # Group by street and calculate aggregate stats
    agg_funcs = {
        'estimated_floors': ['mean', 'max']
    }
    grouped = joined.groupby('TRAM').agg(agg_funcs)
    
    # Flatten MultiIndex columns
    grouped.columns = [f"catastral_bldg_floors_{col[1]}_{buffer_size}m" for col in grouped.columns]
    
    return grouped.reset_index()

features_50m = calculate_building_floors_in_buffer(noise_streets, buildings, 50)
features_100m = calculate_building_floors_in_buffer(noise_streets, buildings, 100)

display(features_50m.head())

In [ ]:
dataset = pd.DataFrame({'street_id': noise_streets['TRAM']})

dataset = dataset.merge(features_50m, left_on='street_id', right_on='TRAM', how='left').drop(columns=['TRAM'])
dataset = dataset.merge(features_100m, left_on='street_id', right_on='TRAM', how='left').drop(columns=['TRAM'])

dataset = dataset.fillna(0)
display(dataset.head())

output_dir = "../data/processed"
os.makedirs(output_dir, exist_ok=True)
dataset.to_csv(os.path.join(output_dir, "catastral_building_floors.csv"), index=False)
print("Exported catastral_building_floors.csv")

In [ ]:
# Temporarily merge the result back to the street geometries for plotting
viz_gdf = noise_streets.merge(dataset[['street_id', 'catastral_bldg_floors_mean_50m']], left_on='TRAM', right_on='street_id')

fig, ax = plt.subplots(figsize=(12, 10))

# Cap at 95th percentile so super tall buildings don't wash out the color map gradient
vmax_floors = viz_gdf['catastral_bldg_floors_mean_50m'].quantile(0.95)

viz_gdf.plot(
    column='catastral_bldg_floors_mean_50m',
    ax=ax,
    cmap='viridis',
    linewidth=1.5,
    legend=True,
    legend_kwds={'label': "Average Number of Floors in 50m Buffer"},
    vmax=vmax_floors
)

ax.set_title("Barcelona Street Network: Average Number of Floors Above Ground", fontsize=16)
ax.set_axis_off()
plt.show()